In [1]:

from pathlib import Path

# Current working directory ko project root par normalize karo
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

ARTIFACT_DIR = PROJECT_DIR / "models"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR = PROJECT_DIR / "data" / "raw"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DIR / "zomato_cleaned.csv"

print("Project root:", PROJECT_DIR)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Models:", ARTIFACT_DIR)
print("Dataset exists:", DATA_PATH.exists())


Project root: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline
Raw data: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\data\raw
Processed data: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\data\processed
Models: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline\models
Dataset exists: True


In [2]:

import sys
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

# Make project root importable
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data_loader import load_dataset
from src.preprocessing import (
    TARGET,
    prepare_model_frame,
    make_preprocessor,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Imports successful!")
print("Project root:", PROJECT_DIR)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")


Imports successful!
Project root: d:\Projects\LEARNINGS\MACHINE LEARNING\Swiftdrop delivery pipeline
Device: cpu


In [3]:

# Load the local raw dataset
df_raw = load_dataset(RAW_DIR / "zomato_cleaned.csv")

# Reuse our existing preprocessing module
df_model = prepare_model_frame(df_raw)

# Separate features and target
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET].astype("float32")

print("Raw dataset shape:", df_raw.shape)
print("Prepared dataset shape:", df_model.shape)
print("Feature count:", X.shape[1])
print("Target:", TARGET)
print("\nMissing values in target:", y.isna().sum())
print("\nFeatures with most missing values:")
print(X.isna().sum().sort_values(ascending=False).head(10))


Raw dataset shape: (38964, 22)
Prepared dataset shape: (38964, 26)
Feature count: 25
Target: Time_taken (min)

Missing values in target: 0

Features with most missing values:
order_time_cos                 1204
order_time_minutes             1204
order_time_sin                 1204
Delivery_person_Ratings        1055
Delivery_person_Age            1019
Delivery_location_longitude       0
Restaurant_latitude               0
Restaurant_longitude              0
Delivery_location_latitude        0
Vehicle_condition                 0
dtype: int64


In [4]:

# Same 60/20/20 split as the current XGBoost training script
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=SEED,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
)

# Fit preprocessing ONLY on training data
preprocessor = make_preprocessor(
    X_train,
    scale_numeric=True,
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse matrices to dense arrays for PyTorch
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()

if hasattr(X_val_processed, "toarray"):
    X_val_processed = X_val_processed.toarray()

if hasattr(X_test_processed, "toarray"):
    X_test_processed = X_test_processed.toarray()

X_train_processed = np.asarray(X_train_processed, dtype=np.float32)
X_val_processed = np.asarray(X_val_processed, dtype=np.float32)
X_test_processed = np.asarray(X_test_processed, dtype=np.float32)

y_train_np = y_train.to_numpy(dtype=np.float32)
y_val_np = y_val.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

print("Training rows:", len(X_train_processed))
print("Validation rows:", len(X_val_processed))
print("Test rows:", len(X_test_processed))
print("Processed feature count:", X_train_processed.shape[1])

print("Training NaNs:", np.isnan(X_train_processed).sum())
print("Validation NaNs:", np.isnan(X_val_processed).sum())
print("Test NaNs:", np.isnan(X_test_processed).sum())


Training rows: 23378
Validation rows: 7793
Test rows: 7793
Processed feature count: 46
Training NaNs: 0
Validation NaNs: 0
Test NaNs: 0


In [5]:

pickup_related = [
    col for col in X.columns
    if "picked" in col.lower()
]

print("Pickup-related model features:", pickup_related)
print("Training rows:", len(X_train_processed))
print("Validation rows:", len(X_val_processed))
print("Processed feature count:", X_train_processed.shape[1])
print("Training NaNs:", np.isnan(X_train_processed).sum())
print("Validation NaNs:", np.isnan(X_val_processed).sum())


Pickup-related model features: []
Training rows: 23378
Validation rows: 7793
Processed feature count: 46
Training NaNs: 0
Validation NaNs: 0


In [6]:
from torch.utils.data import Dataset, DataLoader

class DeliveryDataset(Dataset):
    def __init__(self, features, targets):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32).reshape(-1, 1)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.targets[index]


BATCH_SIZE = 256

train_dataset = DeliveryDataset(X_train_processed, y_train_np)
val_dataset = DeliveryDataset(X_val_processed, y_val_np)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# Inspect one batch
batch_features, batch_targets = next(iter(train_loader))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch feature shape:", batch_features.shape)
print("Batch target shape:", batch_targets.shape)


Training batches: 92
Validation batches: 31
Batch feature shape: torch.Size([256, 46])
Batch target shape: torch.Size([256, 1])


In [7]:
import torch.nn as nn

class DeliveryTimeMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.10),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeliveryTimeMLP(
    input_dim=X_train_processed.shape[1]
).to(device)

print(model)
print("\nDevice:", device)

# Verify the output shape
sample_predictions = model(batch_features.to(device))
print("Prediction batch shape:", sample_predictions.shape)


DeliveryTimeMLP(
  (network): Sequential(
    (0): Linear(in_features=46, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)

Device: cpu
Prediction batch shape: torch.Size([256, 1])


In [8]:
import copy
import json
import time
from pathlib import Path

import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Training configuration
LEARNING_RATE = 1e-3
MAX_EPOCHS = 100
PATIENCE = 10

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-5,
)

best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0
history = []

start_time = time.perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    # ---- Training ----
    model.train()
    train_loss_sum = 0.0
    train_count = 0

    for features, targets in train_loader:
        features = features.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        predictions = model(features)
        loss = criterion(predictions, targets)

        loss.backward()
        optimizer.step()

        batch_count = len(features)
        train_loss_sum += loss.item() * batch_count
        train_count += batch_count

    train_loss = train_loss_sum / train_count

    # ---- Validation ----
    model.eval()
    val_loss_sum = 0.0
    val_count = 0

    with torch.no_grad():
        for features, targets in val_loader:
            features = features.to(device)
            targets = targets.to(device)

            predictions = model(features)
            loss = criterion(predictions, targets)

            batch_count = len(features)
            val_loss_sum += loss.item() * batch_count
            val_count += batch_count

    val_loss = val_loss_sum / val_count

    history.append({
        "epoch": epoch,
        "train_mse": train_loss,
        "val_mse": val_loss,
    })

    # Save the best model based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Train MSE: {train_loss:.4f} | "
            f"Val MSE: {val_loss:.4f}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}.")
        break

# Restore the best validation checkpoint
model.load_state_dict(best_state)

training_seconds = time.perf_counter() - start_time

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation MSE: {best_val_loss:.4f}")
print(f"Training time: {training_seconds:.1f} seconds")


Epoch 001 | Train MSE: 352.9052 | Val MSE: 47.2072
Epoch 005 | Train MSE: 36.1205 | Val MSE: 31.4278
Epoch 010 | Train MSE: 32.2995 | Val MSE: 28.2976
Epoch 015 | Train MSE: 30.2124 | Val MSE: 26.0386
Epoch 020 | Train MSE: 29.3685 | Val MSE: 25.3794
Epoch 025 | Train MSE: 29.1542 | Val MSE: 24.6619
Epoch 030 | Train MSE: 28.4391 | Val MSE: 24.4384
Epoch 035 | Train MSE: 27.7240 | Val MSE: 24.4654
Epoch 040 | Train MSE: 27.6489 | Val MSE: 24.0033
Epoch 045 | Train MSE: 27.4713 | Val MSE: 23.8411
Epoch 050 | Train MSE: 27.4237 | Val MSE: 23.8105
Epoch 055 | Train MSE: 26.5767 | Val MSE: 23.6567
Epoch 060 | Train MSE: 26.2792 | Val MSE: 23.4163
Epoch 065 | Train MSE: 25.9419 | Val MSE: 23.1532
Epoch 070 | Train MSE: 25.5873 | Val MSE: 22.8565
Epoch 075 | Train MSE: 25.3396 | Val MSE: 23.0465
Epoch 080 | Train MSE: 24.1811 | Val MSE: 21.6412
Epoch 085 | Train MSE: 23.5197 | Val MSE: 21.1277
Epoch 090 | Train MSE: 22.9584 | Val MSE: 20.1688
Epoch 095 | Train MSE: 22.3665 | Val MSE: 19.6933

In [9]:

model.eval()

with torch.inference_mode():
    val_predictions = model(
        torch.from_numpy(X_val_processed).to(device)
    ).cpu().numpy().reshape(-1)

    test_predictions = model(
        torch.from_numpy(X_test_processed).to(device)
    ).cpu().numpy().reshape(-1)


def calculate_metrics(y_true, y_pred):
    return {
        "MAE (minutes)": float(
            mean_absolute_error(y_true, y_pred)
        ),
        "RMSE (minutes)": float(
            np.sqrt(mean_squared_error(y_true, y_pred))
        ),
        "R2": float(r2_score(y_true, y_pred)),
    }


validation_metrics = calculate_metrics(y_val_np, val_predictions)
test_metrics = calculate_metrics(y_test_np, test_predictions)

print("Neural Network — Validation metrics")
for name, value in validation_metrics.items():
    print(f"{name}: {value:.6f}")

print("\nNeural Network — Test metrics")
for name, value in test_metrics.items():
    print(f"{name}: {value:.6f}")


Neural Network — Validation metrics
MAE (minutes): 3.464553
RMSE (minutes): 4.378737
R2: 0.777161

Neural Network — Test metrics
MAE (minutes): 3.434547
RMSE (minutes): 4.327530
R2: 0.783153


In [11]:
import joblib
validation_path = ARTIFACT_DIR / "nn_validation_data.npz"

np.savez_compressed(
    validation_path,
    X_val_processed=X_val_processed,
    y_val=y_val_np,
    X_test_processed=X_test_processed,
    y_test=y_test_np,
)

torch.save(
    model.cpu().state_dict(),
    ARTIFACT_DIR / "neural_network_weights.pth",
)

joblib.dump(
    preprocessor,
    ARTIFACT_DIR / "neural_network_preprocessor.joblib",
)

print("Updated NN weights and preprocessor saved.")
print("Validation data:", X_val_processed.shape)
print("Test data:", X_test_processed.shape)


Updated NN weights and preprocessor saved.
Validation data: (7793, 46)
Test data: (7793, 46)


In [13]:
MODELS_DIR = PROJECT_DIR / "models"

In [14]:
import torch

model.load_state_dict(best_state)
model.eval()

MODELS_DIR.mkdir(parents=True, exist_ok=True)

torch.save(
    model.state_dict(),
    MODELS_DIR / "neural_network_weights.pth"
)

print("Model weights saved successfully.")

Model weights saved successfully.


In [16]:
import torch
import numpy as np
from sklearn.metrics import mean_absolute_error

# Current in-memory model predictions
model.eval()
with torch.no_grad():
    current_preds = model(
        torch.from_numpy(X_val_processed).float()
    ).cpu().numpy().reshape(-1)

# Fresh model loaded from the saved weights file
saved_model = DeliveryTimeMLP(X_val_processed.shape[1])
saved_model.load_state_dict(
    torch.load(
        MODELS_DIR / "neural_network_weights.pth",
        map_location="cpu",
        weights_only=True
    )
)
saved_model.eval()

with torch.no_grad():
    saved_preds = saved_model(
        torch.from_numpy(X_val_processed).float()
    ).numpy().reshape(-1)

print("Current in-memory MAE:",
      mean_absolute_error(y_val_np, current_preds))
print("Saved model MAE:",
      mean_absolute_error(y_val_np, saved_preds))
print("Max prediction difference:",
      np.max(np.abs(current_preds - saved_preds)))

Current in-memory MAE: 3.464552640914917
Saved model MAE: 3.464552640914917
Max prediction difference: 0.0
